## Model Training

### Spark Session Setup

In [ ]:
import findspark

findspark.init()

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('modelTraining').getOrCreate()

In [ ]:
#data loading will change once data is hosted on hadoop? loading from local for now
from pyspark import SparkFiles
df = spark.read.parquet('prepped_data.parquet')

### Train Test Split

In [ ]:
train_data, test_data = df.randomSplit([0.7, 0.3], seed = 101)

### Model Importing and Setup

In [ ]:
#import models
from pyspark.ml.classification import NaiveBayes, RandomForestClassifier, LogisticRegression, GBTClassifier

#set up models with default settings
nb = NaiveBayes()
rf = RandomForestClassifier(seed = 101)
lr = LogisticRegression()
gb = GBTClassifier(seed = 101)

In [ ]:
#eval set up
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

acc_evaluator = MulticlassClassificationEvaluator(metricName = 'accuracy')

### Hyperparameter Tuning

In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

#naive bayes
grid = ParamGridBuilder().addGrid(nb.smoothing, [0.0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5]).build()
nb_cv = CrossValidator(estimator = nb, estimatorParamMaps = grid, evaluator = acc_evaluator, numFolds = 5, parallelism = 4, seed = 101)

#random forest
grid = ParamGridBuilder().addGrid(rf.numTrees, range(10, 60, 10)).addGrid(rf.maxDepth, range(1, 11)).build()
rf_cv = CrossValidator(estimator = rf, estimatorParamMaps = grid, evaluator = acc_evaluator, numFolds = 5, parallelism = 4, seed = 101)

#logistic regression
grid = ParamGridBuilder().addGrid(lr.regParam, [0.01, 0.1, 0.5, 1.0]).addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]).build()
lr_cv = CrossValidator(estimator = lr, estimatorParamMaps = grid, evaluator = acc_evaluator, numFolds = 5, parallelism = 4, seed = 101)

#gradiant boosted trees
grid = ParamGridBuilder().addGrid(gb.maxDepth, [3, 5, 7]).addGrid(gb.maxIter, [10, 20, 30]).addGrid(gb.stepSize, [0.05, 0.1, 0.2]).build()
gb_cv = CrossValidator(estimator = gb, estimatorParamMaps = grid, evaluator = acc_evaluator, numFolds = 5, parallelism = 4, seed = 101)

In [ ]:
#run predictions
nb_cv_model = nb_cv.fit(train_data)
nb_cv_pred = nb_cv_model.transform(test_data)

In [ ]:
rf_cv_model = rf_cv.fit(train_data)
rf_cv_pred = rf_cv_model.transform(test_data)

In [ ]:
lr_cv_model = lr_cv.fit(train_data)
lr_cv_pred = lr_cv_model.transform(test_data)

In [ ]:
gb_cv_model = gb_cv.fit(train_data)
gb_cv_pred = gb_cv_model.transform(test_data)

In [ ]:
#results
print(nb_cv_model.bestModel)

nb_accuracy = acc_evaluator.evaluate(nb_cv_pred)
print(f"Naive Bayes Accuracy: {nb_accuracy}")

In [ ]:
print(rf_cv_model.bestModel)

rf_accuracy = acc_evaluator.evaluate(rf_cv_pred)
print(f"Random Forest Accuracy: {rf_accuracy}")

In [ ]:
print(lr_cv_model.bestModel)

lr_accuracy = acc_evaluator.evaluate(lr_cv_pred)
print(f"Logistic Regression Accuracy: {lr_accuracy}")

In [ ]:
print(gb_cv_model.bestModel)

gb_accuracy = acc_evaluator.evaluate(gb_cv_pred)
print(f"Gradient Boosted Accuracy: {gb_accuracy}")

### Confusion Matrices

In [ ]:
from pyspark.mllib.evaluation import MulticlassMetrics

nb_predictionAndLabels = nb_cv_pred.select("prediction", "label").rdd.map(lambda row: (float(row['prediction']), float(row['label'])))

metrics = MulticlassMetrics(nb_predictionAndLabels)
confusion_matrix = metrics.confusionMatrix()
print('Naive Bayes Confusion Matrix:')
print(confusion_matrix.toArray())
#in the format of
#TN FP
#FN TP

In [ ]:
rf_predictionAndLabels = rf_cv_pred.select("prediction", "label").rdd.map(lambda row: (float(row['prediction']), float(row['label'])))

metrics = MulticlassMetrics(rf_predictionAndLabels)
confusion_matrix = metrics.confusionMatrix()
print('Random Forest Confusion Matrix:')
print(confusion_matrix.toArray())

In [ ]:
lr_predictionAndLabels = lr_cv_pred.select("prediction", "label").rdd.map(lambda row: (float(row['prediction']), float(row['label'])))

metrics = MulticlassMetrics(lr_predictionAndLabels)
confusion_matrix = metrics.confusionMatrix()
print('Logistic Regression Confusion Matrix:')
print(confusion_matrix.toArray())

In [ ]:
gb_predictionAndLabels = gb_cv_pred.select("prediction", "label").rdd.map(lambda row: (float(row['prediction']), float(row['label'])))

metrics = MulticlassMetrics(gb_predictionAndLabels)
confusion_matrix = metrics.confusionMatrix()
print('Gradient Boosted Confusion Matrix:')
print(confusion_matrix.toArray())

### Saving Results as Gradio Placeholder

In [ ]:
#save predictions for gradio

#udf to expand features vector and get correct confidence
from pyspark.ml.linalg import VectorUDT
from pyspark.sql.functions import udf, col
from pyspark.sql.types import ArrayType, DoubleType

vector_to_array_udf = udf(lambda vec: vec.toArray().tolist(), ArrayType(DoubleType()))

def get_confidence(probability_vector, prediction):
    return float(probability_vector[int(prediction)])
confidence_udf = udf(get_confidence, DoubleType())

In [ ]:
#save nb
nb_pred_expanded = nb_cv_pred.withColumn('features', vector_to_array_udf(col('features')))
feature_size = len(nb_pred_expanded.select('features').first()['features'])
for i in range(feature_size):
    nb_pred_expanded = nb_pred_expanded.withColumn(f"feature_{i}", col('features')[i])

nb_pred_final = nb_pred_expanded.withColumn('confidence', confidence_udf(col('probability'), col('prediction'))).drop('features', 'probability', 'rawPrediction')

nb_pred_final.coalesce(1).write.csv('nb_predictions.csv', header = True, mode = 'overwrite')

In [ ]:
#save rf
rf_pred_expanded = rf_cv_pred.withColumn('features', vector_to_array_udf(col('features')))
feature_size = len(rf_pred_expanded.select('features').first()['features'])
for i in range(feature_size):
    rf_pred_expanded = rf_pred_expanded.withColumn(f"feature_{i}", col('features')[i])

rf_pred_final = rf_pred_expanded.withColumn('confidence', confidence_udf(col('probability'), col('prediction'))).drop('features', 'probability', 'rawPrediction')

rf_pred_final.coalesce(1).write.csv('rf_predictions.csv', header = True, mode = 'overwrite')

In [ ]:
#save lr
lr_pred_expanded = lr_cv_pred.withColumn('features', vector_to_array_udf(col('features')))
feature_size = len(lr_pred_expanded.select('features').first()['features'])
for i in range(feature_size):
    lr_pred_expanded = lr_pred_expanded.withColumn(f"feature_{i}", col('features')[i])

lr_pred_final = lr_pred_expanded.withColumn('confidence', confidence_udf(col('probability'), col('prediction'))).drop('features', 'probability', 'rawPrediction')

lr_pred_final.coalesce(1).write.csv('lr_predictions.csv', header = True, mode = 'overwrite')

In [ ]:
#save gb
gb_pred_expanded = gb_cv_pred.withColumn('features', vector_to_array_udf(col('features')))
feature_size = len(gb_pred_expanded.select('features').first()['features'])
for i in range(feature_size):
    gb_pred_expanded = gb_pred_expanded.withColumn(f"feature_{i}", col('features')[i])

gb_pred_final = gb_pred_expanded.withColumn('confidence', confidence_udf(col('probability'), col('prediction'))).drop('features', 'probability', 'rawPrediction')

gb_pred_final.coalesce(1).write.csv('gb_predictions.csv', header=True, mode='overwrite')

In [ ]:
#all resulting csv files manually renamed and moved to root project folder for easier use

In [ ]:
import pandas as pd
gb_pred_df = pd.read_csv('gb_predictions.csv')
lr_pred_df = pd.read_csv('lr_predictions.csv')
nb_pred_df = pd.read_csv('nb_predictions.csv')
rf_pred_df = pd.read_csv('rf_predictions.csv')

In [ ]:
gb_pred_df.head()

In [ ]:
new_cols = ['true', 'prediction', 'amt', 'city_pop', 'unix_time', 'category_index', 'confidence']
gb_pred_df.columns = new_cols
lr_pred_df.columns = new_cols
nb_pred_df.columns = new_cols
rf_pred_df.columns = new_cols

In [ ]:
join_cols = ['true', 'amt', 'city_pop', 'unix_time', 'category_index']

gb_trimmed = gb_pred_df[join_cols + ['prediction', 'confidence']].rename(columns = {
    'prediction': 'prediction_gb',
    'confidence': 'confidence_gb'
})

lr_trimmed = lr_pred_df[join_cols + ['prediction', 'confidence']].rename(columns = {
    'prediction': 'prediction_lr',
    'confidence': 'confidence_lr'
})

nb_trimmed = nb_pred_df[join_cols + ['prediction', 'confidence']].rename(columns = {
    'prediction': 'prediction_nb',
    'confidence': 'confidence_nb'
})

rf_trimmed = rf_pred_df[join_cols + ['prediction', 'confidence']].rename(columns = {
    'prediction': 'prediction_rf',
    'confidence': 'confidence_rf'
})

pred_df = gb_trimmed.merge(lr_trimmed, on = join_cols)
pred_df = pred_df.merge(nb_trimmed, on = join_cols)
pred_df = pred_df.merge(rf_trimmed, on = join_cols)

In [ ]:
pred_df.to_pickle('combined_predictions.pkl')

In [ ]:
pred_df.head()